# Resolved spline precompute

Recreate the no-decay resolved Gauss B-spline coefficients used by the `Last_stretch` synthetic-data workflow. The output layout is `mode_<number>/without_decay`, matching `R_splines_arbitrary.h5`.

The next cell locates the repository root and imports only dependencies already used by the current synthetic workflow. Importing `synUtils` loads the existing CHAOS projection and resolution helpers, but does not run the generator.

In [ ]:
import sys
from pathlib import Path

import h5py
import numpy as np
from tqdm.auto import tqdm

working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "src" / "msc_thesis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate the repository root containing src/msc_thesis."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.msc_thesis.paths import CHAOS_RESOL_DIR, FELIX_DIR
from src.msc_thesis.synSetup import times_res, times_res_relative
from src.msc_thesis.synUtils import (
    Component_Load_SV,
    G_Time_Series_Eval,
    P,
    R_Apply_Two_Halves,
)

## Configuration and invariant checks

The next cell fixes the original scientific choices: modes 1–62, spherical-harmonic degree 20, the complete 148-point resolution timeline, and the standard output name. It checks dimensions without creating or modifying any file. A temporary `.incomplete.h5` path is used during generation so a partial result cannot be mistaken for the finished file.

In [ ]:
NMAX = 20
MODE_NUMBERS = tuple(range(1, 63))
OUTPUT_PATH = Path(FELIX_DIR) / "R_splines_arbitrary.h5"
INCOMPLETE_PATH = OUTPUT_PATH.with_name(
    f"{OUTPUT_PATH.stem}.incomplete{OUTPUT_PATH.suffix}"
)
RESOLUTION_PATH = (
    Path(CHAOS_RESOL_DIR) / "CHAOS_Resol_1997_2026_0806_MF.h5"
)

EXPECTED_GAUSS_COEFFICIENTS = NMAX * (NMAX + 2)
EXPECTED_TIME_SAMPLES = len(times_res_relative)
EXPECTED_SPLINES = P.shape[0]
EXPECTED_SPLINE_SHAPE = (
    EXPECTED_SPLINES,
    EXPECTED_GAUSS_COEFFICIENTS,
)

if np.shape(times_res) != np.shape(times_res_relative):
    raise ValueError("Absolute and relative resolution times do not align.")
if P.shape[1] != EXPECTED_TIME_SAMPLES:
    raise ValueError(
        f"P expects {P.shape[1]} time samples, but the current resolution "
        f"timeline contains {EXPECTED_TIME_SAMPLES}."
    )
if EXPECTED_TIME_SAMPLES != 148 or EXPECTED_SPLINE_SHAPE != (64, 440):
    raise ValueError(
        "The current resolution products do not match the conventions used "
        "to create the original precomputed spline file."
    )

## Construct and resolve one mode

The next cell defines the per-mode calculation. `Component_Load_SV` returns the no-decay SV phasor $i\omega g$ in the current implementation, so division by its nonzero eigenvalue recovers the corresponding MF phasor $g$. The MF wave is evaluated from the 1997.1 resolution origin, projected with `P`, and passed through the existing resolution-matrix helper. Defining these functions performs no calculation by itself.

In [ ]:
def load_no_decay_mf_phasor(mode_number, nmax=NMAX):
    """Return the unscaled MF phasor and pure-imaginary eigenvalue."""

    mode_data = Component_Load_SV(mode_number, nmax=nmax)
    eigenvalue = complex(np.asarray(mode_data["eigenvalue"]).item())
    sv_phasor = np.asarray(mode_data["gnm"])

    if not np.isfinite(eigenvalue):
        raise ValueError(f"Mode {mode_number} has a non-finite eigenvalue.")
    if eigenvalue.real != 0.0:
        raise ValueError(
            f"Mode {mode_number} is not using the expected no-decay eigenvalue."
        )
    if np.isclose(abs(eigenvalue), 0.0):
        raise ValueError(
            f"Mode {mode_number} has a zero eigenvalue, so its MF phasor "
            "cannot be recovered from the SV phasor."
        )
    if sv_phasor.shape != (EXPECTED_GAUSS_COEFFICIENTS,):
        raise ValueError(
            f"Mode {mode_number} has SV phasor shape {sv_phasor.shape}; "
            f"expected ({EXPECTED_GAUSS_COEFFICIENTS},)."
        )

    mf_phasor = sv_phasor / eigenvalue
    return mf_phasor, eigenvalue


def resolve_no_decay_mode(mode_number):
    """Return one mode's resolved MF B-spline coefficients."""

    mf_phasor, eigenvalue = load_no_decay_mf_phasor(mode_number)
    mf_time_series = G_Time_Series_Eval(
        mf_phasor,
        eigenvalue,
        times=np.asarray(times_res_relative),
    )

    expected_time_series_shape = (
        EXPECTED_TIME_SAMPLES,
        EXPECTED_GAUSS_COEFFICIENTS,
    )
    if mf_time_series.shape != expected_time_series_shape:
        raise ValueError(
            f"Mode {mode_number} produced time-series shape "
            f"{mf_time_series.shape}; expected {expected_time_series_shape}."
        )

    mf_spline = P @ mf_time_series
    if mf_spline.shape != EXPECTED_SPLINE_SHAPE:
        raise ValueError(
            f"Mode {mode_number} produced spline shape {mf_spline.shape}; "
            f"expected {EXPECTED_SPLINE_SHAPE}."
        )

    resolved_spline = R_Apply_Two_Halves(mf_spline, verbose=False)
    resolved_spline = np.real_if_close(resolved_spline, tol=1000)

    if resolved_spline.shape != EXPECTED_SPLINE_SHAPE:
        raise ValueError(
            f"Mode {mode_number} produced resolved shape "
            f"{resolved_spline.shape}; expected {EXPECTED_SPLINE_SHAPE}."
        )
    if np.iscomplexobj(resolved_spline):
        raise ValueError(
            f"Mode {mode_number} retained non-negligible imaginary values."
        )
    if not np.all(np.isfinite(resolved_spline)):
        raise ValueError(
            f"Mode {mode_number} contains non-finite resolved coefficients."
        )

    return np.asarray(resolved_spline)

## Validate the completed temporary file

The next cell defines a structural validation pass. Before the temporary file is promoted to the standard filename, it must contain exactly one group for each requested mode and exactly one `(64, 440)` dataset named `without_decay` in every group. Defining the function does not open any file.

In [ ]:
def validate_precomputed_file(file_path, mode_numbers=MODE_NUMBERS):
    """Check the group names and no-decay dataset shapes."""

    expected_groups = {f"mode_{mode_number}" for mode_number in mode_numbers}

    with h5py.File(file_path, "r") as h5_file:
        actual_groups = set(h5_file.keys())
        if actual_groups != expected_groups:
            missing = sorted(expected_groups - actual_groups)
            unexpected = sorted(actual_groups - expected_groups)
            raise ValueError(
                f"Unexpected mode groups. Missing: {missing}; "
                f"unexpected: {unexpected}."
            )

        for group_name in sorted(expected_groups):
            group = h5_file[group_name]
            if set(group.keys()) != {"without_decay"}:
                raise ValueError(
                    f"{group_name} does not contain exactly the "
                    "'without_decay' dataset."
                )
            if group["without_decay"].shape != EXPECTED_SPLINE_SHAPE:
                raise ValueError(
                    f"{group_name}/without_decay has shape "
                    f"{group['without_decay'].shape}; expected "
                    f"{EXPECTED_SPLINE_SHAPE}."
                )

## Generate the HDF5 file safely

The next cell defines the full writer. It refuses to overwrite either the finished file or a previous incomplete file, creates the temporary file in exclusive mode, records only the no-decay dataset for each mode, validates the result, and only then moves it to `R_splines_arbitrary.h5`. An interrupted calculation deliberately leaves the `.incomplete.h5` file for diagnosis rather than silently replacing anything.

In [ ]:
def write_resolved_spline_file():
    """Generate and validate the no-decay resolved-spline HDF5 file."""

    output_path = OUTPUT_PATH
    incomplete_path = INCOMPLETE_PATH

    if output_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite the existing output file: {output_path}"
        )
    if incomplete_path.exists():
        raise FileExistsError(
            "Refusing to overwrite a previous incomplete file: "
            f"{incomplete_path}"
        )
    if output_path.parent != incomplete_path.parent:
        raise ValueError(
            "The output and incomplete files must share a directory so the "
            "final rename remains local."
        )

    missing_mode_files = [
        Path(FELIX_DIR) / f"mode_surface_including_gnm_{mode_number}.h5"
        for mode_number in MODE_NUMBERS
        if not (
            Path(FELIX_DIR)
            / f"mode_surface_including_gnm_{mode_number}.h5"
        ).is_file()
    ]
    if missing_mode_files:
        raise FileNotFoundError(
            "Missing source mode files: "
            + ", ".join(str(path) for path in missing_mode_files)
        )

    expected_resolution_dimension = int(np.prod(EXPECTED_SPLINE_SHAPE))
    with h5py.File(RESOLUTION_PATH, "r") as resolution_file:
        resolution_shape = resolution_file["R"].shape
    if resolution_shape != (
        expected_resolution_dimension,
        expected_resolution_dimension,
    ):
        raise ValueError(
            f"Resolution matrix shape {resolution_shape} is incompatible with "
            f"the expected spline shape {EXPECTED_SPLINE_SHAPE}."
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with h5py.File(incomplete_path, "x") as h5_file:
        h5_file.attrs["description"] = (
            "Unscaled, no-decay resolved MF Gauss B-spline coefficients"
        )
        h5_file.attrs["maximum_spherical_harmonic_degree"] = NMAX
        h5_file.attrs["resolution_time_origin_decimal_year"] = float(
            times_res[0]
        )
        h5_file.attrs["resolution_matrix_file"] = RESOLUTION_PATH.name

        for mode_number in tqdm(MODE_NUMBERS, desc="Resolving modes"):
            resolved_spline = resolve_no_decay_mode(mode_number)
            group = h5_file.create_group(f"mode_{mode_number}")
            dataset = group.create_dataset(
                "without_decay",
                data=resolved_spline,
            )
            dataset.attrs["decay_applied"] = False
            dataset.attrs["maximum_spherical_harmonic_degree"] = NMAX
            h5_file.flush()

    validate_precomputed_file(incomplete_path, mode_numbers=MODE_NUMBERS)

    if output_path.exists():
        raise FileExistsError(
            "The output file appeared during generation; the validated result "
            f"has been left at {incomplete_path}."
        )

    incomplete_path.replace(output_path)
    return output_path

## Explicit execution gate

The final cell is intentionally disabled. If reconstruction is ever necessary, first confirm that the original output and any incomplete file are absent, then change `RUN_GENERATOR` to `True` and run the notebook. With the default `False`, this cell performs no generation and writes nothing.

In [ ]:
RUN_GENERATOR = False

if RUN_GENERATOR:
    generated_path = write_resolved_spline_file()
    print(f"Finished writing {generated_path}")
else:
    print(
        "Generator disabled. Set RUN_GENERATOR = True only when reconstruction "
        "is required and the existing output has been confirmed absent."
    )